
# C 방식 (클린 버전): 제품 × 여러명 배치 프롬프트 생성
입력
- `/mnt/data/persona_attributes_weighted.jsonl`
- `/mnt/data/product_info_preprocessed.jsonl`

출력
- `/mnt/data/prompts_C.jsonl`
- `/mnt/data/prompts_C_preview.json`


In [ ]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("/mnt/data/persona_attributes_weighted.jsonl")
PRODUCT_JSONL = Path("/mnt/data/product_info_preprocessed.jsonl")

OUT_JSONL     = Path("/mnt/data/prompts_C.jsonl")
OUT_PREVIEW   = Path("/mnt/data/prompts_C_preview.json")

BATCH_SIZE = 10
ATTR_LIMIT = 40

print("CONFIG loaded.")

In [ ]:

# =============================
# 1) Load data
# =============================
import json

personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            personas.append(json.loads(line))

products = []
with open(PRODUCT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            products.append(json.loads(line))

print("Loaded:", len(personas), "personas /", len(products), "products")

In [ ]:

# =============================
# 2) Helpers
# =============================
from typing import List, Dict, Any

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def format_attributes_for_prompt(attrs: Dict[str, Any], limit:int=40) -> str:
    items = sorted(attrs.items(), key=lambda kv: kv[1].get("weight", 0.0), reverse=True)[:limit]
    lines = []
    for k, vw in items:
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines)

def persona_to_prompt_block(p: Dict[str, Any]) -> str:
    meta = p.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", meta.get("Description",""))
    attrs_str = format_attributes_for_prompt(p.get("attributes", {}), limit=ATTR_LIMIT)

    if (cluster or label or desc):
        cluster_block = "- cluster: " + str(cluster) + "\n" + "- label: " + str(label) + "\n" + "- desc: " + str(desc)
    else:
        cluster_block = "- cluster: N/A"

    # assemble with concatenation to avoid f-string backslash issues
    parts = []
    parts.append("[페르소나]")
    parts.append(f"- id: {p.get('persona_key','')}")
    parts.append("- 속성(가중치 합=1):")
    parts.append(attrs_str)
    parts.append("- 클러스터 컨텍스트:")
    parts.append(cluster_block)
    return "\n".join(parts).strip()

def build_batch_prompt(product: Dict[str, Any], persona_batch: List[Dict[str, Any]]) -> str:
    product_block = product.get("prompt_block") or ""
    persona_blocks = [persona_to_prompt_block(p) for p in persona_batch]
    persona_blocks_str = "\n\n".join(persona_blocks)

    parts = []
    parts.append("[역할]")
    parts.append("당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.")
    parts.append("아래의 \"제품 정보\"와 \"페르소나 목록\"을 바탕으로, 각 페르소나마다")
    parts.append("해당 제품의 구매자 페르소나를 **싱글 턴**으로 완결된 JSON 객체로 생성하세요.")
    parts.append("각 페르소나는 서로 독립적이며, 서로의 정보에 영향을 주지 마세요.")
    parts.append("")
    parts.append("[제품 정보]")
    parts.append(product_block)
    parts.append("")
    parts.append("[페르소나 목록]")
    parts.append(persona_blocks_str)
    parts.append("")
    parts.append("[규칙]")
    parts.append("- '클러스터 컨텍스트'는 페르소나의 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.")
    parts.append("- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.")
    parts.append("- 추석/설, 광고/프로모션/계절성을 반영합니다.")
    parts.append("- **반드시 아래 JSON 스키마(JSON 배열)를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.")
    parts.append("")
    parts.append("[출력 스키마(JSON 배열)]")
    parts.append("[")
    parts.append("  {")
    parts.append('    "persona_id": "p_{{product_id_or_name}}_{{persona_key}}",')
    parts.append(f'    "product_name": "{product.get("product_name","")}",')
    parts.append(f'    "product_id": "{product.get("product_id","")}",')
    parts.append('    "segment_ref": "{{persona_key}}",')
    parts.append('    "attributes": { "{{속성명}}": {"value": "<값>", "weight": <0~1> }, "...": "..." },')
    parts.append('    "purchase_pattern": {')
    parts.append('      "avg_purchase_prob": <0~1>,')
    parts.append('      "avg_purchase_qty": <int>,')
    parts.append('      "seasonality": {"추석": "+x%", "설": "+y%"},')
    parts.append('      "promotion_effect": "광고/프로모션 노출 시 +z%"')
    parts.append('    },')
    parts.append('    "forecast_12mo": {')
    parts.append('      "2024-07": {"prob": <0~1>, "qty": <int>},')
    parts.append('      "...": {},')
    parts.append('      "2025-06": {"prob": <0~1>, "qty": <int>}')
    parts.append('    }')
    parts.append("  },")
    parts.append("  ...")
    parts.append("]")
    return "\n".join(parts).strip()

In [ ]:

# =============================
# 3) Build & save
# =============================
import json
from pathlib import Path

records = []
for prod in products:
    pid_or_name = prod.get("product_id") or (prod.get("product_name","") or "").replace(" ", "_")
    # chunk personas
    for i in range(0, len(personas), BATCH_SIZE):
        batch = personas[i:i+BATCH_SIZE]
        prompt_text = build_batch_prompt({**prod, "product_id_or_name": pid_or_name}, batch)
        records.append({
            "product": {"product_id_or_name": pid_or_name, "product_name": prod.get("product_name")},
            "personas": [{"persona_key": p.get("persona_key")} for p in batch],
            "prompt": prompt_text
        })

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Path(OUT_PREVIEW).write_text(json.dumps(records[:1], ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUT_JSONL, "bytes=", OUT_JSONL.stat().st_size)
len(records)